In [ ]:
import os, gc, math, random
from dataclasses import dataclass
from glob import glob
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
@dataclass
class CFG:
    TRAIN_META_CSV: str = "/kaggle/input/physionet-ecg-image-digitization/train.csv"
    TEST_CSV: str = "/kaggle/input/physionet-ecg-image-digitization/test.csv"
    TRAIN_DIR: str = "/kaggle/input/physionet-ecg-image-digitization/train"
    TEST_DIR: str = "/kaggle/input/physionet-ecg-image-digitization/test"

    OUT_DIR: str = "/kaggle/working"
    WEIGHTS_OUT: str = "/kaggle/working/timm_unet_trace_best.pth"
    SUBMISSION_OUT: str = "/kaggle/working/submission.csv"

    SEED: int = 42
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    AMP: bool = True
    
    ENCODER: str = "tf_efficientnet_b0"
    PRETRAINED_ENCODER: bool = False
    EPOCHS: int = 25
    LR: float = 3e-4
    WD: float = 1e-2
    BATCH: int = 12
    NUM_WORKERS: int = 2
    GRAD_CLIP: float = 5.0
    PHASE1_EPOCHS: int = 10
    PHASE2_LR_MULT: float = 0.35
    PANEL_W: int = 512
    PANEL_H: int = 256
    GAUSS_SIGMA_PX: float = 2.0
    POS_WEIGHT: float = 6.0
    WHITE_THR: int = 245
    DESKEW_MAX_DEG: float = 15.0
    TOP_FRAC: float = 0.70
    INNER_PAD_FRAC_X: float = 0.015
    INNER_PAD_FRAC_Y: float = 0.08
    DP_MAX_STEP: int = 3
    DP_DOWNSCALE_H: float = 0.55
    SMOOTH_WIN: int = 13
    SOFTARGMAX_TEMP: float = 10.0
    WAVE_L1_W: float = 0.22
    SMOOTH_D1_W: float = 0.02
    SMOOTH_D2_W: float = 0.01
    USE_SHIFT_TOL_LOSS: bool = True
    SHIFT_LOSS_W: float = 0.55
    SHIFT_TAU: float = 0.035
    SHIFT_MAX_FRAC: float = 0.010
    ENERGY_K: float = 2e-4
    SNR_MAX_SHIFT_FRAC: float = 0.01
    SNR_MAX_SHIFT_ABS: int = 0
    SNR_SKIP_ENERGY_THR: float = 1e-5
    VALID_FRAC: float = 0.10
    SQPX_MIN: float = 3.0
    SQPX_MAX: float = 40.0
    SQPX_FALLBACK: float = 8.0
    MM_PER_MV: float = 10.0
    USE_TRACE_CHANNEL: bool = True
    TRACE_BH_FRAC: float = 1.0 / 48.0
    TRACE_EDGE_W: float = 0.30
    TRACE_BH_W: float = 0.70

cfg = CFG()

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(cfg.SEED)

DEVICE = torch.device(cfg.DEVICE)
AMP = cfg.AMP and (DEVICE.type == "cuda")
print("Device:", DEVICE, "| AMP:", AMP)

os.makedirs(cfg.OUT_DIR, exist_ok=True)
torch.backends.cudnn.benchmark = True
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

try:
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)

LEAD_ROWS = [
    ["I",   "aVR", "V1", "V4"],
    ["II",  "aVL", "V2", "V5"],
    ["III", "aVF", "V3", "V6"],
]
RHYTHM_LEAD = "II"
LEADS_12 = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]

def lead_position_in_grid(lead: str):
    for r in range(3):
        for c in range(4):
            if LEAD_ROWS[r][c] == lead:
                return (r, c)
    return None

def select_wave_segment_for_panel(lead: str, full_series: np.ndarray) -> np.ndarray:
    if full_series is None or len(full_series) < 8:
        return full_series
    if lead == RHYTHM_LEAD:
        return full_series
    L = len(full_series)
    q = max(8, int(round(L / 4.0)))
    return full_series[:q]

def smooth_1d(x: np.ndarray, w: int) -> np.ndarray:
    if w <= 1:
        return x.astype(np.float32)
    w = int(w)
    if w % 2 == 0:
        w += 1
    pad = w // 2
    xp = np.pad(x.astype(np.float32), (pad, pad), mode="edge")
    ker = np.ones(w, np.float32) / w
    return np.convolve(xp, ker, mode="valid").astype(np.float32)

def rectified_crop_and_deskew(img_bgr: np.ndarray) -> np.ndarray:
    H, W = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    mask = (gray < cfg.WHITE_THR).astype(np.uint8) * 255
    mask = cv2.medianBlur(mask, 5)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        c = max(cnts, key=cv2.contourArea)
        x,y,w,h = cv2.boundingRect(c)
        pad = int(0.01 * min(H,W))
        x0 = max(0, x - pad); y0 = max(0, y - pad)
        x1 = min(W, x + w + pad); y1 = min(H, y + h + pad)
        crop = img_bgr[y0:y1, x0:x1].copy()
    else:
        crop = img_bgr.copy()

    g = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(g, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=120, minLineLength=200, maxLineGap=20)

    angle = 0.0
    if lines is not None:
        angles = []
        for (x1,y1,x2,y2) in lines[:,0]:
            dx, dy = (x2-x1), (y2-y1)
            if abs(dx) < 1:
                continue
            a = np.degrees(np.arctan2(dy, dx))
            if abs(a) <= cfg.DESKEW_MAX_DEG:
                angles.append(a)
        if len(angles) >= 8:
            angle = float(np.median(angles))

    if abs(angle) > 0.15:
        ch, cw = crop.shape[:2]
        M = cv2.getRotationMatrix2D((cw/2, ch/2), angle=-angle, scale=1.0)
        crop = cv2.warpAffine(
            crop, M, (cw, ch),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=(255,255,255)
        )
    return crop

def compute_geom_fixed(rect_bgr: np.ndarray):
    H, W = rect_bgr.shape[:2]
    top_end = int(cfg.TOP_FRAC * H)
    row_h = max(1, top_end // 3)

    bands = []
    for r in range(3):
        y0 = r * row_h
        y1 = (r + 1) * row_h if r < 2 else top_end
        bands.append((y0, y1))
    bands.append((top_end, H))

    x_rects = []
    for r in range(3):
        row_rects = []
        for c in range(4):
            x0 = int(c * W / 4.0)
            x1 = int((c + 1) * W / 4.0)
            row_rects.append((x0, x1))
        x_rects.append(row_rects)

    return {"bands": bands, "x_rects": x_rects}

def extract_panel_with_geometry(rect_bgr: np.ndarray, geom: dict, lead: str):
    H, W = rect_bgr.shape[:2]
    bands = geom["bands"]
    pad_x = int(cfg.INNER_PAD_FRAC_X * W)
    pad_y = int(cfg.INNER_PAD_FRAC_Y * (bands[0][1] - bands[0][0]))
    if lead == RHYTHM_LEAD:
        y0, y1 = bands[3]
        y0 = min(max(0, y0 + pad_y), H-1)
        y1 = max(min(H, y1 - pad_y), y0+1)
        x0, x1 = pad_x, max(pad_x+1, W - pad_x)
        crop = rect_bgr[y0:y1, x0:x1]
        if crop.size == 0:
            return None
        crop = cv2.resize(crop, (cfg.PANEL_W, cfg.PANEL_H), interpolation=cv2.INTER_AREA)
        return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    pos = lead_position_in_grid(lead)
    if pos is None:
        return None
    r, c = pos
    y0, y1 = bands[r]
    x0, x1 = geom["x_rects"][r][c]
    y0 = min(max(0, y0 + pad_y), H-1)
    y1 = max(min(H, y1 - pad_y), y0+1)
    x0 = min(max(0, x0 + pad_x), W-1)
    x1 = max(min(W, x1 - pad_x), x0+1)
    crop = rect_bgr[y0:y1, x0:x1]
    if crop.size == 0:
        return None
    crop = cv2.resize(crop, (cfg.PANEL_W, cfg.PANEL_H), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

def trace_channel(panel_rgb: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(panel_rgb, cv2.COLOR_RGB2GRAY)
    k = int(max(3, (min(cfg.PANEL_H, cfg.PANEL_W) * cfg.TRACE_BH_FRAC)))
    if k % 2 == 0:
        k += 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    bh = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    e = cv2.Canny(gray, 40, 120)
    bh = bh.astype(np.float32) / 255.0
    e = e.astype(np.float32) / 255.0
    t = cfg.TRACE_BH_W * bh + cfg.TRACE_EDGE_W * e
    t = cv2.GaussianBlur(t, (3, 3), 0)
    return np.clip(t, 0.0, 1.0).astype(np.float32)

def _period_from_profile_fft(prof: np.ndarray, sqpx_min: float, sqpx_max: float, fallback: float) -> float:
    prof = prof.astype(np.float32)
    prof = prof - float(prof.mean())
    n = prof.shape[0]
    if n < 64:
        return fallback
    mag = np.abs(np.fft.rfft(prof))
    k_min = int(max(1, math.floor(n / sqpx_max)))
    k_max = int(min(len(mag) - 1, math.ceil(n / sqpx_min)))
    if k_max <= k_min + 1:
        return fallback
    k = k_min + int(np.argmax(mag[k_min:k_max]))
    if k <= 0:
        return fallback
    period = float(n / k)
    if not (sqpx_min <= period <= sqpx_max):
        return fallback
    return period

def estimate_small_square_px_from_panel(panel_rgb: np.ndarray) -> float:
    gray = cv2.cvtColor(panel_rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (3,3), 0)
    hp = cv2.absdiff(gray, cv2.GaussianBlur(gray, (0,0), 3))
    H, W = hp.shape
    x0, x1 = int(0.08 * W), int(0.92 * W)
    y0, y1 = int(0.08 * H), int(0.92 * H)
    hp2 = hp[y0:y1, x0:x1]
    if hp2.size < 64 * 64:
        hp2 = hp
    prof_y = hp2.mean(axis=1)
    prof_x = hp2.mean(axis=0)
    py = _period_from_profile_fft(prof_y, cfg.SQPX_MIN, cfg.SQPX_MAX, cfg.SQPX_FALLBACK)
    px = _period_from_profile_fft(prof_x, cfg.SQPX_MIN, cfg.SQPX_MAX, cfg.SQPX_FALLBACK)
    return float(np.median([py, px]))

def pix_per_mV_from_square_px(square_px: float) -> float:
    return float(square_px * cfg.MM_PER_MV)

def estimate_ppmv_for_record(rect_bgr: np.ndarray, geom: dict) -> float:
    leads_probe = ["I", "II", "V1", "V4", RHYTHM_LEAD]
    sqs = []
    for ld in leads_probe:
        p = extract_panel_with_geometry(rect_bgr, geom, ld)
        if p is None:
            continue
        sq = estimate_small_square_px_from_panel(p)
        if np.isfinite(sq) and (cfg.SQPX_MIN <= sq <= cfg.SQPX_MAX):
            sqs.append(float(sq))
    if len(sqs) == 0:
        sq = cfg.SQPX_FALLBACK
    else:
        sq = float(np.median(sqs))
    return pix_per_mV_from_square_px(sq)

def series_to_resampled_mv(series: np.ndarray, W: int) -> np.ndarray:
    if series is None or len(series) < 8:
        return np.zeros((W,), np.float32)
    x_old = np.linspace(0.0, 1.0, len(series), endpoint=False)
    x_new = np.linspace(0.0, 1.0, W, endpoint=False)
    return np.interp(x_new, x_old, series).astype(np.float32)

def make_trace_heatmap_from_mv(s_mv: np.ndarray, H: int, W: int, sigma_px: float, pix_per_mV: float):
    if s_mv is None or len(s_mv) != W:
        return np.zeros((H, W), np.float32)
    mid = (H - 1) / 2.0
    y = np.clip(mid - s_mv * pix_per_mV, 0, H-1)
    yy = np.arange(H, dtype=np.float32)[:, None]
    heat = np.exp(-0.5 * ((yy - y[None, :]) / float(sigma_px))**2).astype(np.float32)
    return np.clip(heat, 0.0, 1.0)

def dp_trace_path(prob_seg: np.ndarray, max_step: int, downscale_h: float):
    h, w = prob_seg.shape
    if downscale_h < 1.0 and h > 80:
        new_h = max(40, int(h * downscale_h))
        prob_ds = cv2.resize(prob_seg, (w, new_h), interpolation=cv2.INTER_LINEAR)
        scale_back = h / new_h
        h2 = new_h
    else:
        prob_ds = prob_seg
        scale_back = 1.0
        h2 = h
    cost = (1.0 - prob_ds).astype(np.float32)
    dp = np.zeros((h2, w), np.float32)
    back = np.zeros((h2, w), np.int32)
    dp[:, 0] = cost[:, 0]
    back[:, 0] = np.arange(h2)
    for x in range(1, w):
        prev = dp[:, x-1]
        for y in range(h2):
            y0 = max(0, y - max_step)
            y1 = min(h2, y + max_step + 1)
            idx = y0 + int(np.argmin(prev[y0:y1]))
            dp[y, x] = cost[y, x] + prev[idx]
            back[y, x] = idx
    y = int(np.argmin(dp[:, -1]))
    path = np.zeros((w,), np.float32)
    path[-1] = y
    for x in range(w-1, 0, -1):
        y = back[y, x]
        path[x-1] = y
    if scale_back != 1.0:
        path = np.clip(path * scale_back, 0, h - 1)
    return path.astype(np.float32)

def digitize_panel_to_mv(prob_panel: np.ndarray, target_len: int, pix_per_mV: float) -> np.ndarray:
    H, W = prob_panel.shape
    y_path = dp_trace_path(prob_panel, cfg.DP_MAX_STEP, cfg.DP_DOWNSCALE_H)
    y_path = smooth_1d(y_path, cfg.SMOOTH_WIN)
    mid = (H - 1) / 2.0
    s_mv = (mid - y_path) / (pix_per_mV + 1e-6)
    s_mv = s_mv.astype(np.float32)
    x_old = np.linspace(0.0, 1.0, W, endpoint=False)
    x_new = np.linspace(0.0, 1.0, int(target_len), endpoint=False)
    return np.interp(x_new, x_old, s_mv).astype(np.float32)

def ecg_postprocess_mv(s: np.ndarray) -> np.ndarray:
    n = len(s)
    if n < 16:
        return s.astype(np.float32)
    w_base = max(9, (n // 20) | 1)
    base = smooth_1d(s, w_base)
    return (s - base).astype(np.float32)

def snr_db(x: np.ndarray, y: np.ndarray, eps: float = 1e-12) -> float:
    num = float(np.sum(x * x)) + eps
    den = float(np.sum((y - x) * (y - x))) + eps
    return 10.0 * math.log10(num / den)

def best_shift_snr_db_shift_only(x: np.ndarray, y: np.ndarray, max_shift: int = 0) -> float:
    best = -1e18
    n = len(x)
    if n < 8:
        return float("nan")
    for s in range(-max_shift, max_shift + 1):
        if s < 0:
            xs = x[-s:]
            ys = y[: n + s]
        elif s > 0:
            xs = x[: n - s]
            ys = y[s:]
        else:
            xs = x
            ys = y
        if len(xs) < 8:
            continue
        b = float(np.mean(xs - ys))
        ys2 = ys + b
        best = max(best, snr_db(xs, ys2))
    return float(best)

train_meta = pd.read_csv(cfg.TRAIN_META_CSV)
if "id" not in train_meta.columns:
    raise ValueError(f"train.csv missing 'id'. Columns={list(train_meta.columns)}")
train_meta["id"] = train_meta["id"].astype(str)
record_ids = train_meta["id"].tolist()

def pick_train_png(record_id: str):
    folder = os.path.join(cfg.TRAIN_DIR, record_id)
    if not os.path.isdir(folder):
        return None
    p0 = os.path.join(folder, f"{record_id}-0.png")
    if os.path.exists(p0):
        return p0
    pngs = sorted(glob(os.path.join(folder, "*.png")))
    return pngs[0] if pngs else None

def load_waveform(record_id: str):
    csv_path = os.path.join(cfg.TRAIN_DIR, record_id, f"{record_id}.csv")
    if not os.path.exists(csv_path):
        return None
    sig = pd.read_csv(csv_path)
    cols = [c for c in LEADS_12 if c in sig.columns]
    if len(cols) < 6:
        return None
    out = {}
    for c in cols:
        v = sig[c].to_numpy(dtype=np.float32)
        if np.isnan(v).any():
            v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        out[c] = v
    return out

series_map, img_index, bad = {}, {}, []
for rid in tqdm(record_ids, desc="Index train"):
    imgp = pick_train_png(rid)
    s = load_waveform(rid)
    if imgp is None or s is None:
        bad.append(rid); continue
    img_index[rid] = imgp
    series_map[rid] = s

records = sorted(series_map.keys())
print("Train usable records:", len(records))
print("Train missing records:", len(bad), "| example:", bad[:5])

rng = np.random.RandomState(cfg.SEED)
perm = records.copy()
rng.shuffle(perm)
n_valid = max(1, int(cfg.VALID_FRAC * len(perm)))
valid_set = set(perm[:n_valid])
train_records = [r for r in perm if r not in valid_set]
valid_records = [r for r in perm if r in valid_set]
print("Split -> train:", len(train_records), "| valid:", len(valid_records))
_record_cache = {}
def get_record_cached(rid: str):
    if rid in _record_cache:
        return _record_cache[rid]
    img_path = img_index[rid]
    img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise FileNotFoundError(img_path)
    rect = rectified_crop_and_deskew(img_bgr)
    geom = compute_geom_fixed(rect)
    ppmv = estimate_ppmv_for_record(rect, geom)
    pack = {"rect": rect, "geom": geom, "ppmv": float(ppmv)}
    _record_cache[rid] = pack
    return pack
class PanelDataset(Dataset):
    def __init__(self, record_list, split="train"):
        self.samples = []
        for rid in record_list:
            for ld in series_map[rid].keys():
                if ld in LEADS_12:
                    self.samples.append((rid, ld))
        random.shuffle(self.samples)
        self.split = split
        print(f"{split} samples:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        rid, lead = self.samples[idx]
        pack = get_record_cached(rid)
        panel = extract_panel_with_geometry(pack["rect"], pack["geom"], lead)
        if panel is None:
            panel = np.full((cfg.PANEL_H, cfg.PANEL_W, 3), 255, np.uint8)
        if self.split == "train":
            if random.random() < 0.50:
                panel = cv2.convertScaleAbs(panel, alpha=1.0 + random.uniform(-0.12, 0.12),
                                            beta=random.uniform(-10, 10))
            if random.random() < 0.15:
                panel = cv2.GaussianBlur(panel, (3,3), 0)

        full_series = series_map[rid].get(lead, None)
        seg_series  = select_wave_segment_for_panel(lead, full_series)
        s_mv = series_to_resampled_mv(seg_series, cfg.PANEL_W)
        heat = make_trace_heatmap_from_mv(s_mv, cfg.PANEL_H, cfg.PANEL_W, cfg.GAUSS_SIGMA_PX, pack["ppmv"])

        x_rgb = (panel.astype(np.float32) / 255.0).transpose(2,0,1)
        if cfg.USE_TRACE_CHANNEL:
            t = trace_channel(panel)
            x = np.concatenate([x_rgb, t[None, :, :]], axis=0)
        else:
            x = x_rgb

        y = heat[None, :, :]
        ppmv = np.array([pack["ppmv"]], dtype=np.float32)
        return (torch.from_numpy(x),
                torch.from_numpy(y),
                torch.from_numpy(s_mv.astype(np.float32)),
                torch.from_numpy(ppmv))

train_ds = PanelDataset(train_records, split="train")
valid_ds = PanelDataset(valid_records, split="valid")

pin = (DEVICE.type == "cuda")
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH, shuffle=True,
                          num_workers=cfg.NUM_WORKERS, pin_memory=pin, drop_last=True)
valid_loader = DataLoader(valid_ds, batch_size=cfg.BATCH, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=pin, drop_last=False)

class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, p=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, k, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.SiLU(inplace=True)
    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv1 = ConvBNAct(in_ch + skip_ch, out_ch)
        self.conv2 = ConvBNAct(out_ch, out_ch)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x

class TimmUNetHeatmap(nn.Module):
    def __init__(self, encoder_name, in_ch=4, out_ch=1, pretrained=False):
        super().__init__()
        self.encoder = timm.create_model(
            encoder_name,
            features_only=True,
            in_chans=in_ch,
            out_indices=(1,2,3,4),
            pretrained=pretrained,
        )
        chs = self.encoder.feature_info.channels()
        self.bottleneck = nn.Sequential(ConvBNAct(chs[-1], 256), ConvBNAct(256, 256))
        self.dec1 = DecoderBlock(256, chs[2], 128)
        self.dec2 = DecoderBlock(128, chs[1], 64)
        self.dec3 = DecoderBlock(64,  chs[0], 32)
        self.head = nn.Conv2d(32, out_ch, kernel_size=1)

    def forward(self, x):
        in_hw = x.shape[-2:]
        f1, f2, f3, f4 = self.encoder(x)
        x = self.bottleneck(f4)
        x = self.dec1(x, f3)
        x = self.dec2(x, f2)
        x = self.dec3(x, f1)
        x = self.head(x)
        x = F.interpolate(x, size=in_hw, mode="bilinear", align_corners=False)
        return x

in_ch = 4 if cfg.USE_TRACE_CHANNEL else 3
model = TimmUNetHeatmap(cfg.ENCODER, in_ch=in_ch, out_ch=1, pretrained=cfg.PRETRAINED_ENCODER).to(DEVICE)
print("Model:", cfg.ENCODER, "| pretrained:", cfg.PRETRAINED_ENCODER, "| in_ch:", in_ch)

pos_weight = torch.tensor([cfg.POS_WEIGHT], device=DEVICE)
bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def dice_loss_with_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    num = 2.0 * (probs * targets).sum(dim=(2,3))
    den = (probs + targets).sum(dim=(2,3)) + eps
    return (1.0 - (num / den)).mean()

def softargmax_waveform_mv_from_logits(logits: torch.Tensor, ppmv: torch.Tensor) -> torch.Tensor:
    B, _, H, W = logits.shape
    temp = float(cfg.SOFTARGMAX_TEMP)
    p = torch.softmax(logits * temp, dim=2)
    yy = torch.arange(H, device=logits.device, dtype=logits.dtype).view(1,1,H,1)
    y_soft = (p * yy).sum(dim=2).squeeze(1)
    mid = (H - 1) / 2.0
    ppmv = ppmv.view(B, 1).to(dtype=logits.dtype)
    s_pred_mv = (mid - y_soft) / (ppmv + 1e-6)
    return s_pred_mv

def smoothness_priors(s: torch.Tensor):
    d1 = s[:, 1:] - s[:, :-1]
    d2 = s[:, 2:] - 2.0*s[:, 1:-1] + s[:, :-2]
    return d1.abs().mean(), d2.abs().mean()

def shift_tolerant_l1(pred: torch.Tensor, gt: torch.Tensor, max_shift: int, tau: float) -> torch.Tensor:
    B, W = pred.shape
    errs = []
    for s in range(-max_shift, max_shift + 1):
        if s < 0:
            p = pred[:, -s:]
            g = gt[:, :W+s]
        elif s > 0:
            p = pred[:, :W-s]
            g = gt[:, s:]
        else:
            p = pred
            g = gt
        if p.shape[1] < 8:
            continue
        b = (g - p).mean(dim=1, keepdim=True)
        err = (p + b - g).abs().mean(dim=1)
        errs.append(err)
    E = torch.stack(errs, dim=1)
    return (-tau * torch.logsumexp(-E / tau, dim=1)).mean()

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WD)

@torch.no_grad()
def valid_epoch():
    model.eval()
    losses, accs, snrs = [], [], []
    used, skipped = 0, 0
    W = cfg.PANEL_W
    max_shift = cfg.SNR_MAX_SHIFT_ABS if cfg.SNR_MAX_SHIFT_ABS > 0 else int(cfg.SNR_MAX_SHIFT_FRAC * W)
    for x, y_heat, y_series_mv, ppmv in valid_loader:
        x = x.to(DEVICE, non_blocking=True)
        y_heat = y_heat.to(DEVICE, non_blocking=True)
        y_series_mv = y_series_mv.to(DEVICE, non_blocking=True)
        ppmv = ppmv.to(DEVICE, non_blocking=True)
        logits = model(x)
        loss_heat = bce(logits, y_heat) + 0.7 * dice_loss_with_logits(logits, y_heat)
        s_pred_soft_mv = softargmax_waveform_mv_from_logits(logits, ppmv)
        wave_l1 = (s_pred_soft_mv - y_series_mv).abs().mean()
        d1, d2 = smoothness_priors(s_pred_soft_mv)
        st = 0.0
        if cfg.USE_SHIFT_TOL_LOSS:
            st_shift = int(cfg.SHIFT_MAX_FRAC * W)
            st = float(shift_tolerant_l1(s_pred_soft_mv, y_series_mv, max_shift=st_shift, tau=cfg.SHIFT_TAU).item())
        loss = loss_heat + cfg.WAVE_L1_W * wave_l1 + cfg.SMOOTH_D1_W * d1 + cfg.SMOOTH_D2_W * d2
        losses.append(float(loss.detach().cpu().item()))
        prob = torch.sigmoid(logits).detach().cpu().numpy()[:, 0]
        tgt  = y_heat.detach().cpu().numpy()[:, 0]
        gt_s = y_series_mv.detach().cpu().numpy()
        ppmv_np = ppmv.detach().cpu().numpy().reshape(-1)
        for i in range(prob.shape[0]):
            y_pred = prob[i].argmax(axis=0).astype(np.int32)
            y_true = tgt[i].argmax(axis=0).astype(np.int32)
            accs.append(float(np.mean(np.abs(y_pred - y_true) <= 2)))
            pred_mv = digitize_panel_to_mv(prob[i].astype(np.float32), target_len=W, pix_per_mV=float(ppmv_np[i]))
            energy = float(np.mean(gt_s[i] * gt_s[i]))
            if energy < cfg.SNR_SKIP_ENERGY_THR:
                skipped += 1
                continue
            used += 1
            snrs.append(best_shift_snr_db_shift_only(gt_s[i].astype(np.float32),
                                                    pred_mv.astype(np.float32),
                                                    max_shift=max_shift))

    model.train()
    mean_snr = float(np.mean(snrs)) if used > 0 else float("nan")
    print(f"[valid] SNR computed on {used}/{used+skipped} samples (skipped {skipped} low-energy GT)")
    return float(np.mean(losses)), float(np.mean(accs)), mean_snr

best_snr = -1e18
W = cfg.PANEL_W
shift_max = int(cfg.SHIFT_MAX_FRAC * W)

for epoch in range(cfg.EPOCHS):
    if epoch == cfg.PHASE1_EPOCHS:
        for pg in optimizer.param_groups:
            pg["lr"] = pg["lr"] * cfg.PHASE2_LR_MULT
        print(f"[phase2] LR reduced. New LR = {optimizer.param_groups[0]['lr']:.3e}")

    model.train()
    pbar = tqdm(train_loader, desc=f"Train {epoch+1}/{cfg.EPOCHS}")
    running = []
    for x, y_heat, y_series_mv, ppmv in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y_heat = y_heat.to(DEVICE, non_blocking=True)
        y_series_mv = y_series_mv.to(DEVICE, non_blocking=True)
        ppmv = ppmv.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=AMP):
            logits = model(x)
            loss_heat = bce(logits, y_heat) + 0.7 * dice_loss_with_logits(logits, y_heat)
            s_pred_soft_mv = softargmax_waveform_mv_from_logits(logits, ppmv)
            energy = (y_series_mv * y_series_mv).mean(dim=1, keepdim=True)
            w_energy = (energy / (energy + cfg.ENERGY_K)).detach()
            w_energy = w_energy.clamp(0.0, 1.0)
            wave_l1 = (s_pred_soft_mv - y_series_mv).abs().mean(dim=1, keepdim=True)
            wave_l1 = (wave_l1 * w_energy).mean()
            shift_term = 0.0
            if cfg.USE_SHIFT_TOL_LOSS and (epoch >= cfg.PHASE1_EPOCHS):
                shift_term = shift_tolerant_l1(s_pred_soft_mv, y_series_mv, max_shift=shift_max, tau=cfg.SHIFT_TAU)
                shift_term = shift_term
            d1, d2 = smoothness_priors(s_pred_soft_mv)
            loss = loss_heat \
                 + cfg.WAVE_L1_W * wave_l1 \
                 + (cfg.SHIFT_LOSS_W * shift_term if (cfg.USE_SHIFT_TOL_LOSS and epoch >= cfg.PHASE1_EPOCHS) else 0.0) \
                 + cfg.SMOOTH_D1_W * d1 + cfg.SMOOTH_D2_W * d2
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        running.append(float(loss.detach().cpu().item()))
        pbar.set_postfix(loss=float(np.mean(running[-50:])))
    v_loss, v_acc, v_snr = valid_epoch()
    print(f"Epoch {epoch+1}: valid_loss={v_loss:.5f} | trace_acc@2px={v_acc:.4f} | SNR(dB)={v_snr:.3f}")
    
    if (not np.isnan(v_snr)) and (v_snr > best_snr):
        best_snr = v_snr
        torch.save({"model": model.state_dict(), "cfg": cfg.__dict__, "best_snr": best_snr}, cfg.WEIGHTS_OUT)
        print("Saved best weights:", cfg.WEIGHTS_OUT, "| best_snr:", best_snr)

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def load_checkpoint_safely(path: str):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

if os.path.exists(cfg.WEIGHTS_OUT):
    ckpt = load_checkpoint_safely(cfg.WEIGHTS_OUT)
    model.load_state_dict(ckpt["model"], strict=True)
    model.eval()
    print("Loaded best checkpoint:", cfg.WEIGHTS_OUT, "| best_snr:", ckpt.get("best_snr", None))

test_df = pd.read_csv(cfg.TEST_CSV)
test_df["id"] = test_df["id"].astype(str)

_test_cache = {}
def get_test_page(rid: str):
    if rid in _test_cache:
        return _test_cache[rid]
    p = os.path.join(cfg.TEST_DIR, f"{rid}.png")
    img = cv2.imread(p, cv2.IMREAD_COLOR)
    if img is None:
        _test_cache[rid] = None
        return None
    rect = rectified_crop_and_deskew(img)
    geom = compute_geom_fixed(rect)
    ppmv = estimate_ppmv_for_record(rect, geom)
    _test_cache[rid] = {"rect": rect, "geom": geom, "ppmv": float(ppmv)}
    return _test_cache[rid]

rows = []
model.eval()
for rec_id, g in tqdm(test_df.groupby("id", sort=False),
                      total=test_df["id"].nunique(), desc="Infer"):
    pack = get_test_page(rec_id)
    if pack is None:
        for _, r in g.iterrows():
            lead = str(r["lead"])
            n = int(r["number_of_rows"])
            for t in range(n):
                rows.append((f"{rec_id}_{t}_{lead}", 0.0))
        continue
    rect = pack["rect"]
    geom = pack["geom"]
    ppmv = float(pack["ppmv"])
    series_by_lead = {}
    for _, r in g.iterrows():
        lead = str(r["lead"])
        n = int(r["number_of_rows"])
        if lead not in series_by_lead:
            panel = extract_panel_with_geometry(rect, geom, lead)
            if panel is None:
                s = np.zeros((n,), np.float32)
            else:
                x_rgb = (panel.astype(np.float32)/255.0).transpose(2,0,1)
                if cfg.USE_TRACE_CHANNEL:
                    t = trace_channel(panel)
                    x = np.concatenate([x_rgb, t[None, :, :]], axis=0)[None, ...]
                else:
                    x = x_rgb[None, ...]
                xt = torch.from_numpy(x).to(DEVICE)

                with torch.amp.autocast("cuda", enabled=AMP):
                    logits = model(xt)
                    prob = torch.sigmoid(logits)[0,0].detach().float().cpu().numpy()
                s = digitize_panel_to_mv(prob.astype(np.float32), target_len=n, pix_per_mV=ppmv)
                s = ecg_postprocess_mv(s)
            series_by_lead[lead] = s
        s = series_by_lead[lead]
        for t in range(n):
            rows.append((f"{rec_id}_{t}_{lead}", float(s[t])))

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
sub = pd.DataFrame(rows, columns=["id","value"])
sub.to_csv(cfg.SUBMISSION_OUT, index=False)
print("Saved submission:", cfg.SUBMISSION_OUT, "| rows:", len(sub))
print(sub.head(10))
print("Best weights:", cfg.WEIGHTS_OUT)